# **DFC-3.4 - лучший сабмит 14 команды**
**Перед проверкой не забудьте прочитать README.MD!**

Ансамбль моделей - то, что разделяет участников и победителей соревнований (почти всегда). 
Мы не стали стоять в стороне от этого подхода, так что решили создать DFC-3.4 - стекинг DFC-3.3 и DFC-3.2 с мета-модель в лице логистической регрессии (хотели ещё добавить DFC-3.1, но он портил скор, так что убрали)

**Общая информация:**
1) Модель: Logistic Regression как мета-модель для стекинга вероятностей базовых DFC-моделей.
2) Используются OOF-предсказания для честной оценки качества на train.
3) По test усредняются предсказания по фолдам, после чего формируется финальный submission.

## **0. Архитектуры моделей DFC-3.3 и DFC-3.2**

Меня ментор попросил, чтобы в одном ноуте было всё решение, но иначе это превратилось в нечитаемое месиво, так что здесь я прикреплю код только архитектур. Полный pipeline обучения в соответствующих ноутбуках

### **DFC-3.2**

In [41]:
import torch 
from torch import nn

class SEBlock(nn.Module):
    def __init__(self, 
                 in_channels: int, 
                 reduction: int = 16
                 ):
        super().__init__()
        self.squeeze = nn.AdaptiveAvgPool2d(1)
        self.excitation = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels // reduction, in_channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, _, _ = x.size()
        y = self.squeeze(x).view(b, c)
        y = self.excitation(y).view(b, c, 1, 1)
        return x * y.expand_as(x)
    
class SEResidualBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1, reduction: int = 16):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.se = SEBlock(out_channels, reduction)

        if in_channels != out_channels or stride != 1:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = self.shortcut(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        
        out = self.se(out)

        out += identity
        out = self.relu(out)
        return out
    
class DFC_3_2(nn.Module):
    def __init__(self, in_channels: int = 3, num_classes: int = 1, dropout_p: float = 0.2):
        super().__init__()
        self.in_channels = 64

        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
        )

        self.layer1 = self.make_layer(out_channels=64, block_count=3, stride=1)
        self.layer2 = self.make_layer(out_channels=128, block_count=4, stride=2)
        self.layer3 = self.make_layer(out_channels=256, block_count=6, stride=2)
        self.layer4 = self.make_layer(out_channels=512, block_count=3, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        
        self.head = nn.Sequential(
            nn.Dropout(p=dropout_p),
            nn.Linear(512, num_classes)
        )

        self._initialize_weights()

    def make_layer(self, out_channels: int, block_count: int, stride: int):
        layers = [SEResidualBlock(self.in_channels, out_channels, stride)]
        self.in_channels = out_channels

        for _ in range(block_count - 1):
            layers.append(SEResidualBlock(self.in_channels, out_channels, stride=1))

        return nn.Sequential(*layers)
        
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.head(x)
        return x

### **DFC-3.3**

In [42]:
class SEBlock(nn.Module):
    def __init__(self, in_channels: int, reduction: int = 16):
        super().__init__()
        self.squeeze = nn.AdaptiveAvgPool2d(1)
        self.excitation = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels // reduction, in_channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, _, _ = x.size()
        y = self.squeeze(x).view(b, c)
        y = self.excitation(y).view(b, c, 1, 1)
        return x * y.expand_as(x)


class SEResidualBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1, reduction: int = 16):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.se = SEBlock(out_channels, reduction)

        if in_channels != out_channels or stride != 1:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = self.shortcut(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.se(out)

        out += identity
        out = self.relu(out)
        return out


class MultiSRMLayer(nn.Module):
    def __init__(self, clamp_value=3.0):
        super().__init__()

        kernels = torch.tensor([
            [[0, 0, 0, 0, 0],
             [0, 0, 0, 0, 0],
             [0, 1, -2, 1, 0],
             [0, 0, 0, 0, 0],
             [0, 0, 0, 0, 0]],

            [[0, 0, 0, 0, 0],
             [0, -1, 2, -1, 0],
             [0, 2, -4, 2, 0],
             [0, -1, 2, -1, 0],
             [0, 0, 0, 0, 0]],

            [[-1, 2, -2, 2, -1],
             [2, -6, 8, -6, 2],
             [-2, 8, -12, 8, -2],
             [2, -6, 8, -6, 2],
             [-1, 2, -2, 2, -1]]
        ], dtype=torch.float32)

        kernels[0] /= 2.0
        kernels[1] /= 4.0
        kernels[2] /= 12.0

        weight = kernels.unsqueeze(1).repeat(3, 1, 1, 1)   # 9, 1, 5, 5
        self.register_buffer("weight", weight)

        self.clamp = nn.Hardtanh(min_val=-clamp_value, max_val=clamp_value)

    def forward(self, x):
        x = F.conv2d(x, self.weight, bias=None, stride=1, padding=2, groups=3)
        x = self.clamp(x)
        return x


def make_stage(in_channels: int, out_channels: int, block_count: int, stride: int):
    layers = [SEResidualBlock(in_channels, out_channels, stride=stride)]
    for _ in range(block_count - 1):
        layers.append(SEResidualBlock(out_channels, out_channels, stride=1))
    return nn.Sequential(*layers)


class DFC_3_3(nn.Module):
    def __init__(self, num_classes: int = 1, dropout_p: float = 0.2):
        super().__init__()

        self.rgb_stem = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
        )

        self.rgb_layer1 = make_stage(64, 64, block_count=3, stride=1)
        self.rgb_layer2 = make_stage(64, 128, block_count=4, stride=2)
        self.rgb_layer3 = make_stage(128, 256, block_count=6, stride=2)
        self.rgb_layer4 = make_stage(256, 512, block_count=3, stride=2)

        self.srm = MultiSRMLayer()

        self.hf_stem = nn.Sequential(
            nn.Conv2d(9, 32, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
        )

        self.hf_layer1 = make_stage(32, 32, block_count=1, stride=1)
        self.hf_layer2 = make_stage(32, 64, block_count=2, stride=2)
        self.hf_layer3 = make_stage(64, 128, block_count=2, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        self.head = nn.Sequential(
            nn.Dropout(p=dropout_p),
            nn.Linear(512 + 128, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_p),
            nn.Linear(128, num_classes)
        )

        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
    
            elif isinstance(m, nn.BatchNorm2d):
                if m.weight is not None:
                    nn.init.constant_(m.weight, 1)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
    
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rgb = self.rgb_stem(x)
        rgb = self.rgb_layer1(rgb)
        rgb = self.rgb_layer2(rgb)
        rgb = self.rgb_layer3(rgb)
        rgb = self.rgb_layer4(rgb)
        rgb = self.avgpool(rgb)
        rgb = torch.flatten(rgb, 1)

        hf = self.srm(x)
        hf = self.hf_stem(hf)
        hf = self.hf_layer1(hf)
        hf = self.hf_layer2(hf)
        hf = self.hf_layer3(hf)
        hf = self.avgpool(hf)
        hf = torch.flatten(hf, 1)

        feats = torch.cat([rgb, hf], dim=1)
        out = self.head(feats)
        return out

## **1. Импорты**

In [43]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report

In [44]:
import warnings
warnings.filterwarnings("ignore")

## **2. Общие константы**
- `OOF_PATHS` - пути до файлов со всеми oof-предсказаниями
- `TEST_PATHS` - пути до файлов со всеми test-предсказаниями
- `ID_COL` - название столбца с id (оно, как и другие, менялось во время разработки, так что для борьбы с багами зафиксировали все последующие)
- `TARGET_COL` - название таргета
- `OOF_PROB_COL` - стобец с вероятностями дипфейка
- `TEST_PROB_COL` - стобец с вероятностями в тестах
- `N_SPLITS` - количество фолдов
- `RANDOM_STATE` - фиксация случайности

In [45]:
# Если захотите затестить, то пути к данным нужно будет немного поменять (зависит от вашей структуры папок)
# Оригинальные файлы хранятся в /YL_Project_Solution/data
OOF_PATHS = {
    "dfc32": "/Users/daniilkrasnov/Desktop/YL_Project_Solution/data/oof_predictions_dfc3_21m(1).csv",
    "dfc33": "/Users/daniilkrasnov/Desktop/YL_Project_Solution/data/oof_predictions_dfc3_21m.csv",
}

TEST_PATHS = {
    "dfc32": "/Users/daniilkrasnov/Desktop/YL_Project_Solution/data/test_probabilities_dfc3_21m(2).csv",
    "dfc33": "/Users/daniilkrasnov/Desktop/YL_Project_Solution/data/test_probabilities_dfc3_21m.csv",
}

ID_COL = "Id"
TARGET_COL = "IsDeepfake"
OOF_PROB_COL = "oof_probability"
TEST_PROB_COL = "probability"
N_SPLITS = 5
RANDOM_STATE = 42


## **3. Чтение данных**

In [46]:
df = pd.read_csv(OOF_PATHS["dfc32"])
df.head()

,Id,ImageName,IsDeepfake,fold,oof_probability
0,0,0.jpg,0,1,2.099949e-07
1,1,1.jpg,1,2,9.999977e-01
2,2,2.jpg,1,2,9.999359e-01
3,3,3.jpg,0,5,1.339950e-06
4,4,4.jpg,0,3,7.342079e-05


In [47]:
# Загрузка OOF вероятностей
def load_oof(path, model_name):
    df = pd.read_csv(path).copy()
    keep_cols = [c for c in [ID_COL, TARGET_COL, OOF_PROB_COL] if c in df.columns]
    df = df[keep_cols].copy()
    df = df.rename(columns={OOF_PROB_COL: f"prob_{model_name}"})
    return df


# Загрузка тестовых вероятностей
def load_test(path, model_name):
    df = pd.read_csv(path).copy()
    df = df[[ID_COL, TEST_PROB_COL]].copy()
    df = df.rename(columns={TEST_PROB_COL: f"prob_{model_name}"})
    return df


# Добавляем мета-признаки на основе вероятностей
def add_meta_features(df, prob_cols):
    out = df.copy()
    probs = out[prob_cols].values

    out["prob_mean"] = probs.mean(axis=1)
    out["prob_std"] = probs.std(axis=1)
    out["prob_min"] = probs.min(axis=1)
    out["prob_max"] = probs.max(axis=1)
    out["prob_median"] = np.median(probs, axis=1)
    out["prob_range"] = out["prob_max"] - out["prob_min"]
    out["prob_12_gap"] = np.abs(out[prob_cols[0]] - out[prob_cols[1]])

    return out


# Подбираем оптимальный порог, максимизация F1-метрики
def find_best_threshold(y_true, probs, grid=None):
    if grid is None:
        grid = np.arange(0.01, 0.991, 0.005)

    best_thr = 0.5
    best_f1 = -1.0
    best_prec = None
    best_rec = None

    for thr in grid:
        pred = (probs >= thr).astype(int)
        f1 = f1_score(y_true, pred)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = float(thr)
            best_prec = precision_score(y_true, pred, zero_division=0)
            best_rec = recall_score(y_true, pred, zero_division=0)

    return {
        "threshold": best_thr,
        "f1": best_f1,
        "precision": best_prec,
        "recall": best_rec,
    }

In [48]:
# Загрузка oof данных
oof_frames = []
for name, path in OOF_PATHS.items():
    oof_frames.append(load_oof(path, name))

oof_df = oof_frames[0]
for cur in oof_frames[1:]:
    oof_df = oof_df.merge(cur, on=[ID_COL, TARGET_COL], how="inner")

oof_df = oof_df.sort_values(ID_COL).reset_index(drop=True)
prob_cols = [c for c in oof_df.columns if c.startswith("prob_")]
oof_df = add_meta_features(oof_df, prob_cols)

oof_df.head()

,Id,IsDeepfake,prob_dfc32,prob_dfc33,prob_mean,prob_std,prob_min,prob_max,prob_median,prob_range,prob_12_gap
0,0,0,2.099949e-07,2.836553e-10,1.051393e-07,1.048556e-07,2.836553e-10,2.099949e-07,1.051393e-07,2.097112e-07,2.097112e-07
1,1,1,9.999977e-01,1.000000e+00,9.999989e-01,1.132488e-06,9.999977e-01,1.000000e+00,9.999989e-01,2.264977e-06,2.264977e-06
2,2,1,9.999359e-01,9.999481e-01,9.999420e-01,6.139278e-06,9.999359e-01,9.999481e-01,9.999420e-01,1.227856e-05,1.227856e-05
3,3,0,1.339950e-06,2.948776e-06,2.144363e-06,8.044128e-07,1.339950e-06,2.948776e-06,2.144363e-06,1.608826e-06,1.608826e-06
4,4,0,7.342079e-05,3.664401e-06,3.854260e-05,3.487820e-05,3.664401e-06,7.342079e-05,3.854260e-05,6.975639e-05,6.975639e-05


In [49]:
# Загрузка тестовых данных
test_frames = []
for name, path in TEST_PATHS.items():
    test_frames.append(load_test(path, name))

test_df = test_frames[0]
for cur in test_frames[1:]:
    test_df = test_df.merge(cur, on=[ID_COL], how="inner")

test_df = test_df.sort_values(ID_COL).reset_index(drop=True)
test_df = add_meta_features(test_df, prob_cols)

test_df.head()

,Id,prob_dfc32,prob_dfc33,prob_mean,prob_std,prob_min,prob_max,prob_median,prob_range,prob_12_gap
0,0,0.695762,0.852512,0.774137,0.078375,0.695762,0.852512,0.774137,0.156750,0.156750
1,1,0.973842,0.999676,0.986759,0.012917,0.973842,0.999676,0.986759,0.025834,0.025834
2,2,0.999715,0.999760,0.999737,0.000023,0.999715,0.999760,0.999737,0.000045,0.000045
3,3,0.000044,0.000169,0.000107,0.000062,0.000044,0.000169,0.000107,0.000125,0.000125
4,4,0.000006,0.000239,0.000122,0.000117,0.000006,0.000239,0.000122,0.000233,0.000233


## **4. Обучение**

Здесь я обучаю модель логистической регрессии. Дабы побороть проблему дисбаланса, я обучаю их через 5-fold Stratified кросс-валидацию, Balanced class weights

P.S. я пробовал обучать и catboost classifier, но он не дал более хорошего скора, так что логрег

In [50]:
feature_cols = ['prob_dfc32', 'prob_dfc33', 'prob_mean', 'prob_std', 'prob_min', 
                'prob_max', 'prob_median', 'prob_range', 'prob_12_gap']

X = oof_df[feature_cols].values
X_test = test_df[feature_cols].values
y = oof_df[TARGET_COL].values

In [51]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

meta_oof = np.zeros(len(oof_df), dtype=np.float64)
meta_test_folds = []

In [52]:
for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), start=1):
    X_tr, X_val = X[tr_idx], X[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]

    model = LogisticRegression(
        C=0.5,
        class_weight="balanced",
        max_iter=2000,
        solver="liblinear",
        random_state=RANDOM_STATE,
    )
    model.fit(X_tr, y_tr)

    val_probs = model.predict_proba(X_val)[:, 1]
    meta_oof[val_idx] = val_probs

    test_probs = model.predict_proba(X_test)[:, 1]
    meta_test_folds.append(test_probs)

    fold_thr_info = find_best_threshold(y_val, val_probs)

In [ ]:
meta_test = np.mean(meta_test_folds, axis=0)

global_thr_info = find_best_threshold(y, meta_oof)
print("Meta-model OOF threshold info:", global_thr_info)

pred_oof = (meta_oof >= global_thr_info["threshold"]).astype(int)
print("\nClassification report:")
print(classification_report(y, pred_oof))


Meta-model OOF threshold info: {'threshold': 0.8199999999999998, 'f1': 0.9117112752815792, 'precision': 0.9393561267781383, 'recall': 0.8856470588235295}

Classification report:
              precision    recall  f1-score   support

           0       0.98      0.99      0.98     41500
           1       0.94      0.89      0.91      8500

    accuracy                           0.97     50000
   macro avg       0.96      0.94      0.95     50000
weighted avg       0.97      0.97      0.97     50000



In [54]:
meta_model = LogisticRegression(
    C=0.5,
    class_weight="balanced",
    max_iter=3000,
    solver="liblinear",
    random_state=RANDOM_STATE,
)
meta_model.fit(X, y)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",0.5
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:

In [55]:
meta_test_full = meta_model.predict_proba(X_test)[:, 1]
final_thr_info = find_best_threshold(y, meta_oof)
print("\nFinal threshold info:", final_thr_info)


Final threshold info: {'threshold': 0.8199999999999998, 'f1': 0.9117112752815792, 'precision': 0.9393561267781383, 'recall': 0.8856470588235295}


## **5. Сабмит решения**

In [56]:
submission_oof_thr = pd.DataFrame({
    ID_COL: test_df[ID_COL],
    TARGET_COL: (meta_test_full >= global_thr_info["threshold"]).astype(int),
})

In [57]:
submission_oof_thr.to_csv("best_submission.csv", index=False)

print("Saved:")
print("- best_submission.csv")

Saved:
- best_submission.csv
